# 🚀 Robot Safety Shutdown System (Simple Reflex Agent + LangGraph + Groq)

This notebook implements a **Simple Reflex Agent** for robot safety shutdown using:
- LangGraph (Agent workflow)
- Groq API (LLaMA 3.1 8B Instant)
- Gradio UI (Professional Light Theme)

---

In [ ]:
# 1. Install Required Libraries
!pip install langgraph langchain langchain_groq gradio

In [ ]:
# 2. Import Libraries
import os
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict
import gradio as gr

In [ ]:
# 3. Set Groq API Key
os.environ['GROQ_API_KEY'] = 'your_groq_api_key_here'

In [ ]:
# 4. Initialize LLM
llm = ChatGroq(
    model='llama-3.1-8b-instant',
    temperature=0
)

In [ ]:
# Define State
class RobotState(TypedDict):
    temperature: float
    threshold: float
    status: str
    decision: str
    explanation: str

In [ ]:
# 5. Reflex Safety Logic (Core Agent)
def safety_check(state: RobotState):
    temp = state['temperature']
    threshold = state['threshold']
    
    if temp > threshold:
        state['decision'] = 'SHUTDOWN'
        state['status'] = 'CRITICAL'
    else:
        state['decision'] = 'CONTINUE'
        state['status'] = 'SAFE'
    
    return state

In [ ]:
# 6. LLM Explanation Node
def generate_explanation(state: RobotState):
    prompt = f"""
    You are an AI safety engineer.

    Robot Engine Temperature: {state['temperature']}°C
    Threshold: {state['threshold']}°C
    Decision: {state['decision']}

    Provide a professional safety report.
    """

    response = llm.invoke([HumanMessage(content=prompt)])
    state['explanation'] = response.content
    return state

In [ ]:
# 7. Build LangGraph
graph = StateGraph(RobotState)

graph.add_node('safety_check', safety_check)
graph.add_node('llm_explainer', generate_explanation)

graph.set_entry_point('safety_check')
graph.add_edge('safety_check', 'llm_explainer')
graph.add_edge('llm_explainer', END)

app = graph.compile()

In [ ]:
# Run Function
def run_system(temp, threshold):
    state = {
        'temperature': temp,
        'threshold': threshold,
        'status': '',
        'decision': '',
        'explanation': ''
    }
    result = app.invoke(state)
    return result

In [ ]:
# 8. Gradio UI
def ui(temp, threshold):
    result = run_system(temp, threshold)
    
    return f"""
    <div style='background:#f9fafb;padding:20px;border-radius:10px;'>
        <h2>🚀 Robot Safety Monitoring System</h2>
        <p><b>Temperature:</b> {temp} °C</p>
        <p><b>Threshold:</b> {threshold} °C</p>
        <h3 style='color:{'red' if result['decision']=='SHUTDOWN' else 'green'};'>Decision: {result['decision']}</h3>
        <p>Status: {result['status']}</p>
        <div style='background:white;padding:10px;border-radius:8px;'>
            <h4>AI Report</h4>
            <p>{result['explanation']}</p>
        </div>
    </div>
    """

interface = gr.Interface(
    fn=ui,
    inputs=[
        gr.Slider(0,1200,value=500,label='Temperature'),
        gr.Slider(100,1200,value=900,label='Threshold')
    ],
    outputs='html',
    title='Robot Safety Shutdown System'
)

interface.launch()